# 04.01 · Documentos de las sesiones de la Preparatoria CCI

Cada sesión del CCI llega como una carpeta en `data/raw/4. Preparatoria CCI/` con el nombre de
su fecha (por ejemplo, `06 de noviembre de 2026`) y los PDF de la agenda, numerados como en la
convocatoria (`01. …pdf`, `02. …pdf`). Este notebook prepara esos documentos para leerlos con
Azure en el `04.02`:

| Caso | Qué se guarda |
|---|---|
| PDF con texto | El texto de cada página, marcado con `[PÁGINA n]` |
| PDF escaneado (sin texto, como los RC y DEC de líneas bolivianas) | Una imagen JPG por página, que Azure lee en el `04.02` |

No llama a la API. La salida es
`data/interim/4. Preparatoria CCI/04.01_documentos/documentos.csv`, una fila por documento, y
las imágenes en `04.01_documentos/paginas/<sesión>/<documento>/`.

## 1. LOAD LIBRARIES

pymupdf lee el texto de los PDF y renderiza las páginas escaneadas. Las rutas se toman de
`caf_input_cci_preparatoria/config.py`.

In [ ]:
from datetime import date
from hashlib import sha256
from pathlib import Path
import re
import sys

from IPython.display import display
import pandas as pd
import pymupdf

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

### Rutas y parámetros

Un PDF se trata como escaneado si tiene menos de `minimo_caracteres_por_pagina` caracteres de
texto por página en promedio. Las páginas escaneadas se renderizan a `dpi_imagenes` puntos por
pulgada: 130 basta para leer las tablas de los reportes de crédito.

In [ ]:
carpeta_sesiones = config.RAW_PREPARATORIA
carpeta_salida = config.INTERIM / "4. Preparatoria CCI" / "04.01_documentos"
archivo_salida = carpeta_salida / "documentos.csv"
carpeta_paginas = carpeta_salida / "paginas"

minimo_caracteres_por_pagina = 200
dpi_imagenes = 130

### Sesiones y documentos

Cada subcarpeta es una sesión. Se listan sus PDF en orden de nombre; los demás archivos se
ignoran y se informan.

In [ ]:
sesiones = sorted(carpeta for carpeta in carpeta_sesiones.iterdir() if carpeta.is_dir())
pdfs = [pdf for sesion in sesiones for pdf in sorted(sesion.glob("*.pdf"))]
otros = [
    archivo
    for sesion in sesiones
    for archivo in sesion.iterdir()
    if archivo.is_file() and archivo.suffix.lower() != ".pdf" and not archivo.name.startswith(".")
]
print(f"{len(sesiones)} sesiones y {len(pdfs)} PDF.")
for archivo in otros:
    print(f"Se ignora (no es PDF): {archivo.relative_to(carpeta_sesiones)}")

## 3. PROCESS DATA

### Fecha de la sesión y número en la agenda

La fecha sale del nombre de la carpeta (`06 de noviembre de 2026`) y será la fecha de la fila
en el Excel. El número en la agenda sale del prefijo del archivo (`01.`); si no lo tiene, se
deja vacío y se avisa en el `04.05`.

In [ ]:
MESES = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "setiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12,
}


def fecha_sesion(nombre_carpeta):
    encontrado = re.fullmatch(r"\s*(\d{1,2}) de (\w+) de (\d{4})\s*", nombre_carpeta.lower())
    if encontrado is None or encontrado[2] not in MESES:
        return ""
    return date(int(encontrado[3]), MESES[encontrado[2]], int(encontrado[1])).isoformat()


def numero_agenda(nombre_archivo):
    encontrado = re.match(r"\s*(\d{1,2})\s*[.\-_ ]", nombre_archivo)
    return str(int(encontrado[1])) if encontrado else ""


for sesion in sesiones:
    print(f"{sesion.name!r} -> {fecha_sesion(sesion.name) or 'FECHA NO RECONOCIDA'}")

### Lectura de cada PDF

`leer_pdf` devuelve el texto con marcas de página y, si el PDF es escaneado, renderiza sus
páginas como JPG. La huella (SHA-256 del archivo) identifica el documento: si el PDF no cambia,
los notebooks siguientes reutilizan sus respuestas de Azure. Las imágenes se rehacen solo si
faltan.

In [ ]:
def leer_pdf(ruta):
    documento = pymupdf.open(ruta)
    paginas = [pagina.get_text().strip() for pagina in documento]
    caracteres = sum(len(texto) for texto in paginas)
    escaneado = caracteres < minimo_caracteres_por_pagina * max(1, len(paginas))
    carpeta_imagenes = ""
    if escaneado:
        destino = carpeta_paginas / ruta.parent.name / ruta.stem
        destino.mkdir(parents=True, exist_ok=True)
        for numero, pagina in enumerate(documento, start=1):
            imagen = destino / f"pagina_{numero:03d}.jpg"
            if not imagen.exists():
                pixeles = pagina.get_pixmap(dpi=dpi_imagenes)
                imagen.write_bytes(pixeles.tobytes("jpg", jpg_quality=80))
        carpeta_imagenes = str(destino.relative_to(carpeta_salida))
    texto = "\n\n".join(
        f"[PÁGINA {numero}]\n{contenido}" for numero, contenido in enumerate(paginas, start=1)
    )
    return {
        "sesion": ruta.parent.name,
        "fecha_sesion": fecha_sesion(ruta.parent.name),
        "numero_agenda": numero_agenda(ruta.name),
        "archivo": ruta.name,
        "huella": sha256(ruta.read_bytes()).hexdigest(),
        "paginas": len(paginas),
        "caracteres": caracteres,
        "escaneado": escaneado,
        "carpeta_imagenes": carpeta_imagenes,
        "texto": "" if escaneado else texto,
    }

In [ ]:
documentos = pd.DataFrame([leer_pdf(pdf) for pdf in pdfs])
documentos.index = pd.RangeIndex(1, len(documentos) + 1)
columnas_vista = ["fecha_sesion", "numero_agenda", "archivo", "paginas", "caracteres", "escaneado"]
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70):
    display(documentos[columnas_vista])

### Controles

Se avisa si una sesión no tiene fecha reconocible, si falta el número en la agenda o si dos
documentos de una misma sesión tienen el mismo número.

In [ ]:
avisos = []
for fila in documentos.itertuples():
    if not fila.fecha_sesion:
        avisos.append(f"{fila.sesion}: el nombre de la carpeta no es una fecha reconocible")
    if not fila.numero_agenda:
        avisos.append(f"{fila.archivo}: sin número de agenda al inicio del nombre")
repetidos = documentos[
    documentos["numero_agenda"].ne("")
    & documentos.duplicated(["sesion", "numero_agenda"], keep=False)
]
for fila in repetidos.itertuples():
    avisos.append(f"{fila.sesion}: número {fila.numero_agenda} repetido ({fila.archivo})")
print("\n".join(avisos) if avisos else "Sin avisos.")
print(f"Escaneados (se leerán como imágenes): {documentos['escaneado'].sum()}")

## 4. SAVE DATA

`documentos.csv` tiene una fila por PDF:

| Columnas | Contenido |
|---|---|
| `sesion`, `fecha_sesion`, `numero_agenda` | Carpeta de la sesión, su fecha (AAAA-MM-DD) y el número en la agenda |
| `archivo`, `huella` | Nombre del PDF y SHA-256 del archivo |
| `paginas`, `caracteres`, `escaneado` | Tamaño del PDF y si se leerá como imágenes |
| `carpeta_imagenes` | Carpeta de las páginas JPG, relativa a `04.01_documentos/` (solo escaneados) |
| `texto` | Texto con marcas `[PÁGINA n]` (vacío en los escaneados) |

In [ ]:
carpeta_salida.mkdir(parents=True, exist_ok=True)
documentos.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print(f"{len(documentos)} documentos guardados en {archivo_salida.relative_to(ROOT)}")